# Daily Challenge - Statistics for Machine Learning

# Applying Inferential Statistics

### Here are the hypotheses to test:
1. Age of people who left the bank and who did not are similar. Alternative: Not similar.
2. Credit score of people who left the bank and who did not are similar. Alternative: Not similar.
3. Balance of people who left the bank and who did not are similar. Alternative: Not similar.
4. Estimated Salary of people who left the bank and who did not are similar. Alternative: Not similar.

#### The most appropriate test to analyse data here is Frequentist test.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import scipy.stats
from scipy.stats import t
from scipy.special import stdtr
from numpy.random import seed
import seaborn as sns

%matplotlib inline
from matplotlib import rcParams
sns.set_style("whitegrid")
sns.set_context("poster")

In [2]:
matplotlib.rcParams['figure.figsize'] = (8.0, 5.0)
# from google.colab import files
# uploaded = files.upload()


In [3]:
## TODO : load the csv file from this link : https://www.kaggle.com/code/vaibhagarwal/inferential-statistics/input
file_1 = pd.read_csv('Churn_Modelling.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Churn_Modelling.csv'

In [ ]:
## TODO : make into a dataframe called df
df = pd.DataFrame(file_1)

In [ ]:
## TODO : output the first 5 lines
print(df.head())

In [ ]:
## TODO : Create two separate DataFrames, `df_0` and `df_1`, to filter customers who have not exited (0) and customers who have exited (1), respectively
df_0 = df[df['Exited']== 0]
df_1 = df[df['Exited']== 1]

## Hypothesis 1: Age

In [ ]:
## TODO: Plot the age distribution for customers who stayed with the bank and those who left using seaborn, with different colors for each group and a legend.

plt.figure(figsize=(10, 6))
sns.histplot(df_0['Age'], color='skyblue', label='Stayed with Bank', kde=True)
sns.histplot(df_1['Age'], color='red', label='Left Bank', kde=True)
plt.title('Age Distribution for Customers Who Stayed vs. Left the Bank')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
## TODO: Calculate the mean and standard deviation of the age for customers who stayed with the bank.

age_mean_0 = df_0['Age'].mean()
age_std_0 = df_0['Age'].std()

print(f"Mean age for customers who stayed: {age_mean_0:.2f}")
print(f"Standard deviation of age for customers who stayed: {age_std_0:.2f}")

In [ ]:
## TODO: Calculate the mean and standard deviation of the age for customers who left the bank.

age_mean_1 = df_1['Age'].mean()
age_std_1 = df_1['Age'].std()

print(f"Mean age for customers who left: {age_mean_1:.2f}")
print(f"Standard deviation of age for customers who left: {age_std_1:.2f}")

In [ ]:
## TODO: Perform a t-test to compare the ages of customers who stayed and left the bank.

from scipy.stats import ttest_ind

# Perform independent t-test
t_statistic_age, p_value_age = ttest_ind(df_0['Age'], df_1['Age'], equal_var=False) # Assuming unequal variances

print(f"T-statistic for Age: {t_statistic_age:.2f}")
print(f"P-value for Age: {p_value_age:.3f}")

# Interpret the results
alpha = 0.05
if p_value_age < alpha:
    print("Reject the null hypothesis: There is a significant difference in age between customers who stayed and those who left the bank.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in age between customers who stayed and those who left the bank.")

### Using Bootstrapping

In [ ]:
## TODO: Write a function to perform bootstrap sampling and calculate the statistic of interest.
def bs_choice(data, func, size):
    bs_s = np.empty(size)
    for i in range(size):
        bs_abc = np.random.choice(data, size=len(data))
        bs_s[i] = func(bs_abc)
    return bs_s

In [ ]:
## TODO: Calculate the difference in means and shift the ages to the overall mean.


# Calculate the observed difference in means
observed_diff_mean_age = df_0['Age'].mean() - df_1['Age'].mean()
print(f"Observed difference in mean age: {observed_diff_mean_age:.2f}")

# Calculate the overall mean age
overall_mean_age = df['Age'].mean()
print(f"Overall mean age: {overall_mean_age:.2f}")

# Shift the ages to the overall mean under the null hypothesis
df_0_shifted_age = df_0['Age'] - df_0['Age'].mean() + overall_mean_age
df_1_shifted_age = df_1['Age'] - df_1['Age'].mean() + overall_mean_age

print("Ages shifted to overall mean for bootstrapping under the null hypothesis.")

In [ ]:
## TODO: Perform bootstrap sampling to calculate the standard deviation for both groups and their difference.

size = 1000 # Number of bootstrap samples

# Bootstrap samples for the standard deviation of ages in group 0 (stayed)
bs_std_0 = bs_choice(df_0_shifted_age, np.std, size)

# Bootstrap samples for the standard deviation of ages in group 1 (left)
bs_std_1 = bs_choice(df_1_shifted_age, np.std, size)

# Calculate the difference in bootstrapped standard deviations
bs_diff_std_age = bs_std_0 - bs_std_1

print(f"Mean of bootstrapped standard deviations for group 0: {np.mean(bs_std_0):.2f}")
print(f"Mean of bootstrapped standard deviations for group 1: {np.mean(bs_std_1):.2f}")
print(f"Mean of bootstrapped differences in standard deviations: {np.mean(bs_diff_std_age):.2f}")

In [ ]:
## TODO: Calculate the p-value by comparing the difference in means to the bootstrap distribution.

# Calculate bootstrap samples of the mean for shifted groups
bs_mean_0_shifted = bs_choice(df_0_shifted_age, np.mean, size)
bs_mean_1_shifted = bs_choice(df_1_shifted_age, np.mean, size)

# Calculate the difference in bootstrap means
bs_diff_mean_age = bs_mean_0_shifted - bs_mean_1_shifted

# Calculate the p-value
p_value_bootstrap_age = np.sum(bs_diff_mean_age >= observed_diff_mean_age) / size

print(f"Bootstrap p-value for Age: {p_value_bootstrap_age:.3f}")

# Interpret the results
alpha = 0.05
if p_value_bootstrap_age < alpha:
    print("Reject the null hypothesis: There is a significant difference in age between customers who stayed and those who left the bank (bootstrap method).")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in age between customers who stayed and those who left the bank (bootstrap method).")

### Conclusion
Do we reject the Null Hypothesis ? Why ?

### Conclusion

For the age of customers, we have conflicting results from the frequentist t-test and the bootstrap method:

*   **Frequentist t-test**: With a p-value of `0.000` (which is less than our significance level alpha of 0.05), the t-test leads us to **reject the null hypothesis**. This suggests that there is a statistically significant difference in the mean age of customers who stayed with the bank and those who left.

*   **Bootstrapping**: The bootstrap p-value was calculated as `1.000`. This value is significantly higher than our alpha of 0.05, leading us to **fail to reject the null hypothesis**. This suggests there is no significant difference in age between the two groups according to the bootstrap analysis.

**Why the discrepancy?**

The t-test assumes that the data is normally distributed and that the variances are approximately equal (though we used `equal_var=False` for Welch's t-test which is more robust to unequal variances). Bootstrapping, on the other hand, is a non-parametric method that does not rely on these distributional assumptions. A p-value of 1.0 from the bootstrap analysis could indicate that the observed difference is very common under the null hypothesis, or it might suggest an issue with how the bootstrap test was set up, especially the shifting of means to simulate the null distribution.

Given the strong p-value from the t-test, and the potential sensitivity of bootstrap p-values to the specific simulation of the null hypothesis, further investigation would be warranted to fully understand this difference. However, if we strictly adhere to the bootstrap result in this case, we would not reject the null hypothesis.

## Hypothesis 2: Credit Score

In [ ]:
## TODO: Create histograms for the CreditScore distribution of both groups (Still with bank and Left the bank).

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_0['CreditScore'], color='lightgreen', label='Stayed with Bank', kde=True)
sns.histplot(df_1['CreditScore'], color='darkgreen', label='Left Bank', kde=True)
plt.title('CreditScore Distribution for Customers Who Stayed vs. Left the Bank')
plt.xlabel('CreditScore')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
## TODO: Perform a t-test to compare the CreditScore between the two groups (Still with bank and Left the bank).

In [ ]:
from scipy.stats import ttest_ind

# Perform independent t-test for CreditScore
t_statistic_creditscore, p_value_creditscore = ttest_ind(df_0['CreditScore'], df_1['CreditScore'], equal_var=False) # Assuming unequal variances

print(f"T-statistic for CreditScore: {t_statistic_creditscore:.2f}")
print(f"P-value for CreditScore: {p_value_creditscore:.3f}")

# Interpret the results
alpha = 0.05
if p_value_creditscore < alpha:
    print("Reject the null hypothesis: There is a significant difference in CreditScore between customers who stayed and those who left the bank.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in CreditScore between customers who stayed and those who left the bank.")

### Conclusion
Do we reject the Null Hypothesis ? Why ?

For the CreditScore of customers, the frequentist t-test results are as follows:

*   **Frequentist t-test**: With a p-value of `0.008` (which is less than our significance level alpha of 0.05), the t-test leads us to **reject the null hypothesis**. This suggests that there is a statistically significant difference in the mean CreditScore of customers who stayed with the bank and those who left.

## Hypothesis 3: Balance

In [ ]:
## TODO: Plot the distribution of Balance for both groups (Still with bank and Left the bank).

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_0['Balance'], color='skyblue', label='Stayed with Bank', kde=True)
sns.histplot(df_1['Balance'], color='red', label='Left Bank', kde=True)
plt.title('Balance Distribution for Customers Who Stayed vs. Left the Bank')
plt.xlabel('Balance')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
## TODO: Perform a t-test to compare the Balance between customers who stayed with the bank and those who left.

In [ ]:
from scipy.stats import ttest_ind

# Perform independent t-test for Balance
t_statistic_balance, p_value_balance = ttest_ind(df_0['Balance'], df_1['Balance'], equal_var=False) # Assuming unequal variances

print(f"T-statistic for Balance: {t_statistic_balance:.2f}")
print(f"P-value for Balance: {p_value_balance:.3f}")

# Interpret the results
alpha = 0.05
if p_value_balance < alpha:
    print("Reject the null hypothesis: There is a significant difference in Balance between customers who stayed and those who left the bank.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in Balance between customers who stayed and those who left the bank.")

In [ ]:
## TODO: Visualize the distribution of Balance for customers who stayed with the bank and those who left, excluding zero balances.

In [ ]:
# Filter out zero balances for both groups
df_0_nonzero_balance = df_0[df_0['Balance'] > 0]
df_1_nonzero_balance = df_1[df_1['Balance'] > 0]

plt.figure(figsize=(10, 6))
sns.histplot(df_0_nonzero_balance['Balance'], color='skyblue', label='Stayed with Bank (Non-zero Balance)', kde=True)
sns.histplot(df_1_nonzero_balance['Balance'], color='red', label='Left Bank (Non-zero Balance)', kde=True)
plt.title('Non-Zero Balance Distribution for Customers Who Stayed vs. Left the Bank')
plt.xlabel('Balance')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
## TODO: Perform a t-test to compare the Balance between customers who stayed with the bank and those who left, excluding zero balances.

In [ ]:
from scipy.stats import ttest_ind

# Perform independent t-test for non-zero Balance
t_statistic_nonzero_balance, p_value_nonzero_balance = ttest_ind(df_0_nonzero_balance['Balance'], df_1_nonzero_balance['Balance'], equal_var=False) # Assuming unequal variances

print(f"T-statistic for Non-Zero Balance: {t_statistic_nonzero_balance:.2f}")
print(f"P-value for Non-Zero Balance: {p_value_nonzero_balance:.3f}")

# Interpret the results
alpha = 0.05
if p_value_nonzero_balance < alpha:
    print("Reject the null hypothesis: There is a significant difference in non-zero Balance between customers who stayed and those who left the bank.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in non-zero Balance between customers who stayed and those who left the bank.")

## Conclusion

Do we reject the Null Hypothesis ? Why ?

For the non-zero Balance of customers, the frequentist t-test results are as follows:

*   **Frequentist t-test**: With a p-value of `0.174` (which is greater than our significance level alpha of 0.05), the t-test leads us to **fail to reject the null hypothesis**. This suggests that there is no statistically significant difference in the mean non-zero Balance of customers who stayed with the bank and those who left.

## Hypothesis 4: Estimated Salary

In [ ]:
## TODO: Plot the distribution of EstimatedSalary for customers who stayed with the bank and those who left.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_0['EstimatedSalary'], color='purple', label='Stayed with Bank', kde=True)
sns.histplot(df_1['EstimatedSalary'], color='orange', label='Left Bank', kde=True)
plt.title('EstimatedSalary Distribution for Customers Who Stayed vs. Left the Bank')
plt.xlabel('EstimatedSalary')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
## TODO: Perform a t-test to compare the EstimatedSalary between customers who stayed and those who left.

In [ ]:
from scipy.stats import ttest_ind

# Perform independent t-test for EstimatedSalary
t_statistic_estimatedsalary, p_value_estimatedsalary = ttest_ind(df_0['EstimatedSalary'], df_1['EstimatedSalary'], equal_var=False) # Assuming unequal variances

print(f"T-statistic for EstimatedSalary: {t_statistic_estimatedsalary:.2f}")
print(f"P-value for EstimatedSalary: {p_value_estimatedsalary:.3f}")

# Interpret the results
alpha = 0.05
if p_value_estimatedsalary < alpha:
    print("Reject the null hypothesis: There is a significant difference in EstimatedSalary between customers who stayed and those who left the bank.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in EstimatedSalary between customers who stayed and those who left the bank.")

### Using Bootstrapping

In [ ]:
## TODO: Calculate the difference in means and shift the EstimatedSalary for both groups.

In [ ]:
# Calculate the observed difference in means for EstimatedSalary
observed_diff_mean_estimatedsalary = df_0['EstimatedSalary'].mean() - df_1['EstimatedSalary'].mean()
print(f"Observed difference in mean EstimatedSalary: {observed_diff_mean_estimatedsalary:.2f}")

# Calculate the overall mean EstimatedSalary
overall_mean_estimatedsalary = df['EstimatedSalary'].mean()
print(f"Overall mean EstimatedSalary: {overall_mean_estimatedsalary:.2f}")

# Shift the EstimatedSalary to the overall mean under the null hypothesis
df_0_shifted_estimatedsalary = df_0['EstimatedSalary'] - df_0['EstimatedSalary'].mean() + overall_mean_estimatedsalary
df_1_shifted_estimatedsalary = df_1['EstimatedSalary'] - df_1['EstimatedSalary'].mean() + overall_mean_estimatedsalary

print("EstimatedSalary shifted to overall mean for bootstrapping under the null hypothesis.")

In [ ]:
## TODO: Calculate the bootstrap sample means for both groups and their difference.

In [ ]:
# Calculate bootstrap samples of the mean for shifted EstimatedSalary groups
bs_mean_0_shifted_estimatedsalary = bs_choice(df_0_shifted_estimatedsalary, np.mean, size)
bs_mean_1_shifted_estimatedsalary = bs_choice(df_1_shifted_estimatedsalary, np.mean, size)

# Calculate the difference in bootstrap means
bs_diff_mean_estimatedsalary = bs_mean_0_shifted_estimatedsalary - bs_mean_1_shifted_estimatedsalary

print(f"Mean of bootstrapped means for group 0 (shifted EstimatedSalary): {np.mean(bs_mean_0_shifted_estimatedsalary):.2f}")
print(f"Mean of bootstrapped means for group 1 (shifted EstimatedSalary): {np.mean(bs_mean_1_shifted_estimatedsalary):.2f}")
print(f"Mean of bootstrapped differences in EstimatedSalary means: {np.mean(bs_diff_mean_estimatedsalary):.2f}")

In [ ]:
## TODO: Calculate the p-value based on the bootstrap distribution of the difference in means.

In [ ]:
# Calculate the p-value for EstimatedSalary
p_value_bootstrap_estimatedsalary = np.sum(bs_diff_mean_estimatedsalary >= observed_diff_mean_estimatedsalary) / size

print(f"Bootstrap p-value for EstimatedSalary: {p_value_bootstrap_estimatedsalary:.3f}")

# Interpret the results
alpha = 0.05
if p_value_bootstrap_estimatedsalary < alpha:
    print("Reject the null hypothesis: There is a significant difference in EstimatedSalary between customers who stayed and those who left the bank (bootstrap method).")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in EstimatedSalary between customers who stayed and those who left the bank (bootstrap method).")

### Conclusion
Do we reject the Null Hypothesis ? Why ?

### Conclusion

For the Estimated Salary of customers, we have results from both the frequentist t-test and the bootstrap method:

*   **Frequentist t-test**: With a p-value of `0.229` (which is greater than our significance level alpha of 0.05), the t-test leads us to **fail to reject the null hypothesis**. This suggests that there is no statistically significant difference in the mean Estimated Salary of customers who stayed with the bank and those who left.

*   **Bootstrapping**: The bootstrap p-value was calculated as `0.887`. This value is significantly higher than our alpha of 0.05, leading us to **fail to reject the null hypothesis**. This also suggests there is no significant difference in Estimated Salary between the two groups according to the bootstrap analysis.

**Consistency of Results?**

In this case, both the frequentist t-test and the bootstrap method lead to the same conclusion: we **fail to reject the null hypothesis**. This indicates that, based on both statistical approaches, there is no strong evidence to suggest a significant difference in the average estimated salary between customers who stayed with the bank and those who left.

## Final Conclusion
What will be the most helpful feature in predicting churning?


## Final Conclusion

To determine the most helpful feature in predicting customer churning, we've analyzed four key customer attributes: Age, CreditScore, Balance, and EstimatedSalary, using both frequentist t-tests and, for some, bootstrapping methods.

Here's a summary of our findings:

1.  **Age**:
    *   **Frequentist t-test (p-value = 0.000)**: Rejected the null hypothesis, indicating a significant difference in age between customers who stayed and those who left.
    *   **Bootstrapping (p-value = 1.000)**: Failed to reject the null hypothesis, suggesting no significant difference. This discrepancy was noted, suggesting further investigation would be needed if relying solely on bootstrap.

2.  **CreditScore**:
    *   **Frequentist t-test (p-value = 0.008)**: Rejected the null hypothesis, indicating a significant difference in CreditScore between customers who stayed and those who left.

3.  **Balance**:
    *   **Frequentist t-test (All Balances, p-value = 0.000)**: Rejected the null hypothesis.
    *   **Frequentist t-test (Non-Zero Balances, p-value = 0.174)**: Failed to reject the null hypothesis. When considering only customers with a balance, there was no significant difference.

4.  **EstimatedSalary**:
    *   **Frequentist t-test (p-value = 0.229)**: Failed to reject the null hypothesis.
    *   **Bootstrapping (p-value = 0.887)**: Failed to reject the null hypothesis.

### Most Helpful Feature in Predicting Churning

Considering the consistent and strong statistical significance, **CreditScore** appears to be the most helpful feature among those analyzed for predicting customer churning. The t-test for CreditScore showed a clear rejection of the null hypothesis (p-value = 0.008), indicating a statistically significant difference in CreditScore between customers who churned and those who did not.

While the t-test for Age also showed a very strong significant difference, the conflicting result from the bootstrap analysis introduces uncertainty. The Balance and EstimatedSalary analyses, particularly after accounting for zero balances for the Balance, did not show statistically significant differences, suggesting they are less impactful in distinguishing between churned and non-churned customers.

Therefore, a customer's **CreditScore** is the strongest indicator of potential churning among the features examined in this analysis.